In [1]:
"""ReportLab executive report builder."""
from __future__ import annotations
from pathlib import Path
from typing import Any
from xml.sax.saxutils import escape
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak

def build_pdf(results:dict, chart_paths:dict, output_path:str|Path)->str:
    """Build a professional report with 15 distinct sections and page numbers."""
    path=Path(output_path); path.parent.mkdir(parents=True,exist_ok=True)
    styles=getSampleStyleSheet(); styles.add(ParagraphStyle(name="Small",parent=styles["BodyText"],fontSize=8,leading=10))
    doc=SimpleDocTemplate(str(path),pagesize=A4,rightMargin=1.6*cm,leftMargin=1.6*cm,topMargin=1.5*cm,bottomMargin=1.5*cm)
    story=[]; k=results.get("kpis",{})
    def heading(t): story.extend([Paragraph(escape(t),styles["Heading1"]),Spacer(1,6)])
    def body(t): story.extend([Paragraph(escape(str(t)),styles["BodyText"]),Spacer(1,8)])
    def section(title,text): heading(title); body(text)
    story.append(Paragraph("AI Business Analyst Executive Report",styles["Title"])); body("Deterministic Pandas analytics with grounded multi-agent interpretation.")
    sections=[("1. Executive Summary",f"Status: {results.get('status','unknown')}. Revenue: {k.get('total_revenue','unavailable')}; orders: {k.get('total_orders','unavailable')}."),
    ("2. Data Sources",results.get("data_sources", "Online Retail dataset, annual report PDF, and Open-Meteo.")),
    ("3. Data Quality",results.get("quality_summary", "Quality checks and cleaning decisions are recorded in the pipeline state.")),
    ("4. Key KPIs",f"Revenue {k.get('total_revenue','unavailable')}; orders {k.get('total_orders','unavailable')}; AOV {k.get('average_order_value','unavailable')}; customers {k.get('unique_customers','unavailable')}."),
    ("5. Sales Analysis",results.get("sales_analysis","Monthly trends and order activity were calculated with Pandas.")),
    ("6. Product Analysis",results.get("product_analysis","Top-product rankings exclude non-product service lines where identified.")),
    ("7. Customer Analysis",results.get("customer_analysis","Customer concentration and Pareto segmentation were calculated deterministically.")),
    ("8. Country Analysis",results.get("country_analysis","Country revenue and order rankings were calculated deterministically.")),
    ("9. Anomaly Analysis",results.get("anomaly_analysis","IQR and robust z-score findings are statistical anomalies, not fraud claims.")),
    ("10. Annual Report Insights",results.get("annual_report_insights","No annual-report evidence supplied.")),
    ("11. Weather/API Insights",results.get("weather_insights","Weather is exploratory association only, not causal evidence.")),
    ("12. AI-generated Business Insights",results.get("insights","Interpretations are grounded in supplied calculations and citations.")),
    ("13. Recommendations",results.get("recommendations","Recommendations should be validated by business owners.")),
    ("14. Limitations",results.get("limitations","LLM, API, location-mismatch, and source-context limitations apply.")),
    ("15. Conclusion",results.get("conclusion","The pipeline completed deterministic analysis and artifact generation."))]
    for title,text in sections: section(title,text)
    if chart_paths:
        story.append(PageBreak()); heading("Charts")
        for name,p in chart_paths.items():
            if Path(p).exists(): story.extend([Paragraph(escape(name.replace('_',' ').title()),styles["Heading2"]),Image(p,width=16*cm,height=8.2*cm),Spacer(1,8)])
    def footer(canvas,doc):
        canvas.saveState(); canvas.setFont("Helvetica",8); canvas.setFillColor(colors.grey); canvas.drawRightString(19*cm,0.8*cm,f"Page {doc.page}"); canvas.restoreState()
    doc.build(story,onFirstPage=footer,onLaterPages=footer)
    return str(path)
